# Classic EVRPTW

**What is this?** A *classic* EVRP with time windows: one depot, many customers, and charging stations. The instance is built on a real road network (OpenStreetMap) for a city you choose.

**How to use this notebook**
- Run cells from top to bottom the first time. After that, you can re-run only the parts you change (e.g. after editing the depot or customer count).
- The map plots are optional previews: you can call them after city, depot, customers, and stations to *see* the instance grow.
- **Customers — pick one mode only**
  - `num_customers` only → the library *generates* customers (buildings, demand, time windows).
  - `customer_csv_path` set → the library *loads* your file; set `num_customers` to `None` in the config (as in the code below). The number of customers is then the number of rows in the CSV.

**Prerequisites:** Python environment with the package installed, and internet if the road network is not already cached.

In [ ]:
# Prefer: pip install -e . (repo root) in this kernel. Fallback if import fails:
import sys
from pathlib import Path
for _p in (Path.cwd() / "src", Path.cwd().parent / "src"):
    if _p.is_dir() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# Use the project environment; from the repo root run: pip install -e .
import evrp_instance_generator_framework as evrp
from evrp_instance_generator_framework.export.instance_export import export_instance
from evrp_instance_generator_framework.visualization import (
    map_benchmark_interactive,
    map_city_roads_interactive,
    map_city_roads_with_depot_interactive,
    map_services_interactive,
)

print("Using package from:", evrp.__file__)
print("Maps below use Folium (interactive Leaflet). No matplotlib plots in this notebook.")


In [ ]:
# This notebook shows maps with Folium only (no `%matplotlib`).
# If embedded maps look blank in Jupyter: File -> Trust Notebook.


## City and road network

**`G` (the graph):** a directed road network for your place. All driving distances and times in the instance are computed on this graph.

**`flat_terrain=True`:** faster prep; set to `False` if you need elevation for energy models. The next cells use the same `G` everywhere.

In [ ]:
city = "Casablanca"
country = "Morocco"
G = evrp.prepare_city_road_network(city, country, flat_terrain=True)
map_city_roads_interactive(G)

## Depot (warehouse / start point)

**`depot_lat`, `depot_lon`:** the *business* location (address or map pin). The engine *snaps* it to the nearest legal road node for routing: that is what `depot_snap_max_dist_m` in the config controls (max allowed snap distance in metres).

You can change these coordinates and re-run the depot plot to check the site.

In [ ]:
depot_lat = 33.573
depot_lon = -7.590
map_city_roads_with_depot_interactive(G, depot_lat, depot_lon)


## How many customers (if you are *not* using a CSV)

Use this only when `customer_csv_path` is `None`. The library will create that many customer *stops* with demand and time windows.

In [ ]:
num_customers = 80


## How many charging stations

These are **selected** from real/synthetic charger candidates near customers and the depot—not random dots. If you set `num_stations` to 0, no public charging nodes are added (depending on variant rules).

In [ ]:
num_stations = 20


## Vehicle (EV) features

These parameters describe *one* vehicle type used for energy and feasibility checks (battery size, mass, drag, driver style, etc.). Defaults are a generic passenger EV; tune them to match your experiment.

In [ ]:
ev_features = evrp.EVFeatures(battery_capacity_kwh=75.0, driver_behavior="passive")


## Optional customer CSV input

Use this when you already exported or prepared customers in the **same format** as the generator (same columns as a full export).

- **`customer_csv_path = None`** → ignore CSV; use `num_customers` above.
- **Path to a `.csv` file** → put `None` for `num_customers` inside `GenerationConfig` (see next code cell). The framework reads `customer_csv_path` from config and replaces generated customers automatically.

Your CSV must include road snap info (`movement_node_id`, `snap_distance_m`) consistent with **this** graph `G`. If you moved to another city or re-downloaded the graph, regenerate or re-snap customer coordinates.

In [ ]:
customer_csv_path = None  # example: "my_customers.csv"

## Build customer set and preview map

**`GenerationConfig`** holds *all* knobs: city, depot, counts, patterns (`customer_pattern`: clustered / random / mix), time-window style (`time_window_tightness`), seed, optional CSV path, etc.

**`customer_generation_result`** is the bundle after the *customer phase*: it contains the movement graph, depot snap, bbox, RNG, and the list **`customer_generation_result.customers`**. Despite the historical name “phase”, think of it as **everything ready before stations**.

The plot shows roads + depot + customers (stations not built yet).

In [ ]:
if "customer_csv_path" not in globals():
    customer_csv_path = None
config = evrp.GenerationConfig(
    variant="classic_evrptw",
    city=city,
    country=country,
    depot_lat=depot_lat,
    depot_lon=depot_lon,
    depot_snap_max_dist_m=500.0,  # max distance (m) to snap depot facility to the road graph
    seed=42,  # reproducibility for random parts of generation
    num_customers=num_customers if customer_csv_path is None else None,
    customer_csv_path=customer_csv_path,
    num_stations=num_stations,
    customer_pattern="rc",  # "c" clustered, "r" random, "rc" mix
    time_window_tightness="medium",  # "wide" | "medium" | "tight"
)
# Runs: prepare graph + depot snap + customer creation (or CSV swap inside the library).
customer_generation_result = evrp.generate_customers_phase(config, ev_features, movement_graph=G)
map_services_interactive(
    customer_generation_result.movement_graph,
    depot_lat,
    depot_lon,
    customers=customer_generation_result.customers,
    stations=None,
)


When using CSV, columns must match (names and types):  
`id`, `lat`, `lon`, `movement_node_id`, `snap_distance_m`, `demand`, `service_time_s`, `parking_time_s`, `time_open_s`, `time_close_s`.

Tip: export a generated instance once and reuse its customer table as a template.

## Select charging stations and preview map

**`generate_stations_phase`** takes `customer_generation_result` and returns a list **`selected_stations`**. Selection uses customer and depot locations so chargers are spread in a useful way, not arbitrary.

Re-run this cell after changing customer count or CSV to refresh stations.

In [ ]:
selected_stations = evrp.generate_stations_phase(customer_generation_result)
map_services_interactive(
    customer_generation_result.movement_graph,
    depot_lat,
    depot_lon,
    customers=customer_generation_result.customers,
    stations=selected_stations,
)


## Generate final benchmark instance

**`finalize_benchmark_instance`** merges customers + stations into **`instance`** (`BenchmarkInstance`): metadata, graph, service nodes, feasibility summary, etc.

Here `compute_matrices=False` skips large distance/time matrices (faster). Set to `True` if you need full matrices in memory. `run_energy_feasibility=True` runs energy-related checks when possible.

In [ ]:
instance = evrp.finalize_benchmark_instance(
    customer_generation_result,
    selected_stations,
    compute_matrices=False,
    run_energy_feasibility=True,
)
map_benchmark_interactive(instance)


## Export files

**`export_instance`** writes datasets under a folder (here JSON). Use this for benchmarks, solvers, or papers. Change `fmt` to `"csv"` if you prefer spreadsheets.

In [ ]:
out_dir = export_instance(instance, output_dir="benchmark_export_classic", fmt="json")
print("Exported to:", out_dir)
